# Pooled Model Comparison Across Datasets

This notebook pools model comparison results across multiple datasets using:
1. Summed AIC (total AIC across all subjects in pooled datasets)
2. Rank-based aggregation (mean rank across datasets)
3. Meta-analytic pooling of ΔAIC (inverse-variance weighted)

In [1]:
import os
import json
import numpy as np
import pandas as pd
from scipy import stats
from jaxcmr.helpers import find_project_root

In [ ]:
# Configuration
fit_tag = "rerun_best_of_1"
fit_dir = "projects/repfr/results/fits/"

# Dataset groupings
free_recall_datasets = ["LohnasKahana2014", "Lohnas2025", "BroitmanKahana2024", "HowardKahana2005"]
serial_recall_datasets = ["RepeatedRecallsKahanaJacobs2000", "RepeatedRecallsGordonRanschburg2021"] 
all_datasets = free_recall_datasets + serial_recall_datasets

model_names = [
    "WeirdCMRNoStop",
    "NoReinstateCMRNoStop",
    "DistinctContextsCMRNoStop",
    "BasePositionalCMRNoStop",
    "FullPositionalCMRNoStop",
    # "McfReinfPositionalCMRNoStop",
    # "MfcReinfPositionalCMRNoStop",
    # "FullReinfPositionalCMRNoStop",
    # "SimpleFullReinfPositionalCMRNoStop",
    "BlendPositionalCMRNoStop",
]

model_titles = [
    "WeirdCMR",
    "NoReinstateCMR",
    "DistinctContextsCMR",
    "BasePositionalCMR",
    "FullPositionalCMR",
    # "MCFReinfPositionalCMR",
    # "MFCReinfPositionalCMR",
    # "FullReinfPositionalCMR",
    # "SimpleFullReinfPositionalCMR",
    "BlendPositionalCMR",
]

In [3]:
def load_results(data_name, model_name):
    """Load fitting results for a dataset-model pair."""
    fit_path = os.path.join(
        find_project_root(), fit_dir,
        f"{data_name}_{model_name}_{fit_tag}.json"
    )
    with open(fit_path) as f:
        result = json.load(f)
    return result

def get_aic_per_subject(result):
    """Compute AIC for each subject: AIC = 2k + 2*NLL."""
    fitness = np.array(result["fitness"])  # NLL per subject (top-level key)
    k = len(result["free"])  # number of free parameters
    return 2 * k + 2 * fitness

def get_total_aic(result):
    """Compute total AIC across all subjects."""
    return np.sum(get_aic_per_subject(result))

def get_mean_aic(result):
    """Compute mean AIC per subject."""
    return np.mean(get_aic_per_subject(result))

In [4]:
# Load all results
all_results = {}
for data_name in all_datasets:
    all_results[data_name] = {}
    for model_name, model_title in zip(model_names, model_titles):
        all_results[data_name][model_title] = load_results(data_name, model_name)

## Method 1: Summed AIC Across Datasets

In [5]:
def compute_summed_aic(datasets, results_dict):
    """Compute total summed AIC across specified datasets for each model."""
    summed_aic = {}
    for model_title in model_titles:
        total = 0
        for data_name in datasets:
            total += get_total_aic(results_dict[data_name][model_title])
        summed_aic[model_title] = total
    return summed_aic

def summed_aic_table(datasets, results_dict, label=""):
    """Create a table of summed AIC results."""
    summed = compute_summed_aic(datasets, results_dict)
    df = pd.DataFrame({
        "Model": list(summed.keys()),
        "Summed AIC": list(summed.values())
    })
    df = df.sort_values("Summed AIC").reset_index(drop=True)
    min_aic = df["Summed AIC"].min()
    df["ΔAIC"] = df["Summed AIC"] - min_aic
    df["Relative Likelihood"] = np.exp(-0.5 * df["ΔAIC"])
    df["AIC Weight"] = df["Relative Likelihood"] / df["Relative Likelihood"].sum()
    return df

In [6]:
print("=" * 60)
print("SUMMED AIC: FREE RECALL DATASETS")
print("=" * 60)
print(f"Datasets: {free_recall_datasets}\n")
df_free = summed_aic_table(free_recall_datasets, all_results)
print(df_free.to_string(index=False))

SUMMED AIC: FREE RECALL DATASETS
Datasets: ['LohnasKahana2014', 'Lohnas2025']

              Model    Summed AIC       ΔAIC  Relative Likelihood    AIC Weight
  BasePositionalCMR 226270.089161   0.000000         1.000000e+00  9.999998e-01
 BlendPositionalCMR 226301.478153  31.388992         1.527450e-07  1.527449e-07
DistinctContextsCMR 226457.793217 187.704056         1.740132e-41  1.740132e-41
  FullPositionalCMR 226886.805878 616.716717        1.206887e-134 1.206887e-134
           WeirdCMR 226982.530052 712.440891        1.974359e-155 1.974359e-155
     NoReinstateCMR 227208.060402 937.971241        2.099582e-204 2.099582e-204


In [7]:
print("=" * 60)
print("SUMMED AIC: SERIAL RECALL DATASETS")
print("=" * 60)
print(f"Datasets: {serial_recall_datasets}\n")
df_serial = summed_aic_table(serial_recall_datasets, all_results)
print(df_serial.to_string(index=False))

SUMMED AIC: SERIAL RECALL DATASETS
Datasets: ['RepeatedRecallsKahanaJacobs2000', 'RepeatedRecallsGordonRanschburg2021']

              Model    Summed AIC         ΔAIC  Relative Likelihood    AIC Weight
  FullPositionalCMR 131041.232086     0.000000         1.000000e+00  1.000000e+00
  BasePositionalCMR 131264.149292   222.917206         3.927750e-49  3.927750e-49
 BlendPositionalCMR 131506.346497   465.114410        1.003897e-101 1.003897e-101
     NoReinstateCMR 136720.806854  5679.574768         0.000000e+00  0.000000e+00
           WeirdCMR 136805.670837  5764.438751         0.000000e+00  0.000000e+00
DistinctContextsCMR 152829.752747 21788.520660         0.000000e+00  0.000000e+00


In [8]:
print("=" * 60)
print("SUMMED AIC: ALL DATASETS")
print("=" * 60)
print(f"Datasets: {all_datasets}\n")
df_all = summed_aic_table(all_datasets, all_results)
print(df_all.to_string(index=False))

SUMMED AIC: ALL DATASETS
Datasets: ['LohnasKahana2014', 'Lohnas2025', 'RepeatedRecallsKahanaJacobs2000', 'RepeatedRecallsGordonRanschburg2021']

              Model    Summed AIC         ΔAIC  Relative Likelihood   AIC Weight
  BasePositionalCMR 357534.238453     0.000000         1.000000e+00 1.000000e+00
 BlendPositionalCMR 357807.824650   273.586197         3.904021e-60 3.904021e-60
  FullPositionalCMR 357928.037964   393.799511         3.072718e-86 3.072718e-86
           WeirdCMR 363788.200890  6253.962437         0.000000e+00 0.000000e+00
     NoReinstateCMR 363928.867256  6394.628803         0.000000e+00 0.000000e+00
DistinctContextsCMR 379287.545963 21753.307510         0.000000e+00 0.000000e+00


## Method 2: Rank-Based Aggregation

In [9]:
def compute_ranks_per_dataset(results_dict):
    """Compute AIC-based ranks for each model within each dataset."""
    ranks = {}
    for data_name in all_datasets:
        # Get mean AIC per model for this dataset
        model_aic = {
            model_title: get_mean_aic(results_dict[data_name][model_title])
            for model_title in model_titles
        }
        # Convert to ranks (1 = best)
        sorted_models = sorted(model_aic.keys(), key=lambda x: model_aic[x])
        ranks[data_name] = {model: rank + 1 for rank, model in enumerate(sorted_models)}
    return ranks

def rank_aggregation_table(results_dict):
    """Create a table showing ranks across datasets and mean rank."""
    ranks = compute_ranks_per_dataset(results_dict)

    data = []
    for model_title in model_titles:
        row = {"Model": model_title}
        for data_name in all_datasets:
            short_name = data_name.replace("RepeatedRecalls", "").replace("Kahana", "K").replace("Gordon", "G").replace("Ranschburg", "R")
            row[short_name] = ranks[data_name][model_title]
        row["Mean Rank"] = np.mean([ranks[dn][model_title] for dn in all_datasets])
        row["Mean Rank (FR)"] = np.mean([ranks[dn][model_title] for dn in free_recall_datasets])
        row["Mean Rank (SR)"] = np.mean([ranks[dn][model_title] for dn in serial_recall_datasets])
        data.append(row)

    df = pd.DataFrame(data)
    df = df.sort_values("Mean Rank").reset_index(drop=True)
    return df

In [10]:
print("=" * 60)
print("RANK-BASED AGGREGATION")
print("=" * 60)
df_ranks = rank_aggregation_table(all_results)
print(df_ranks.to_string(index=False))

RANK-BASED AGGREGATION
              Model  LohnasK2014  Lohnas2025  KJacobs2000  GR2021  Mean Rank  Mean Rank (FR)  Mean Rank (SR)
  BasePositionalCMR            2           1            2       2       1.75             1.5             2.0
 BlendPositionalCMR            1           2            3       3       2.25             1.5             3.0
  FullPositionalCMR            4           6            1       1       3.00             5.0             1.0
DistinctContextsCMR            3           3            6       6       4.50             3.0             6.0
           WeirdCMR            5           4            5       5       4.75             4.5             5.0
     NoReinstateCMR            6           5            4       4       4.75             5.5             4.0


## Method 3: Meta-Analytic Pooling of ΔAIC

In [11]:
def compute_pairwise_delta_aic_stats(results_dict, data_name, ref_model):
    """Compute mean ΔAIC and SE for each model vs reference model."""
    ref_aic = get_aic_per_subject(results_dict[data_name][ref_model])

    stats_dict = {}
    for model_title in model_titles:
        if model_title == ref_model:
            stats_dict[model_title] = {"mean": 0, "se": 0, "n": len(ref_aic)}
            continue
        model_aic = get_aic_per_subject(results_dict[data_name][model_title])
        delta = model_aic - ref_aic  # positive = model is worse
        stats_dict[model_title] = {
            "mean": np.mean(delta),
            "se": np.std(delta, ddof=1) / np.sqrt(len(delta)),
            "n": len(delta)
        }
    return stats_dict

def meta_analytic_pooling(results_dict, datasets, ref_model):
    """Pool ΔAIC estimates across datasets using inverse-variance weighting."""
    pooled = {}

    for model_title in model_titles:
        if model_title == ref_model:
            pooled[model_title] = {"pooled_mean": 0, "pooled_se": 0, "ci_lower": 0, "ci_upper": 0}
            continue

        means = []
        weights = []

        for data_name in datasets:
            stats = compute_pairwise_delta_aic_stats(results_dict, data_name, ref_model)
            if stats[model_title]["se"] > 0:
                means.append(stats[model_title]["mean"])
                weights.append(1 / (stats[model_title]["se"] ** 2))

        if len(weights) > 0:
            weights = np.array(weights)
            means = np.array(means)
            pooled_mean = np.sum(weights * means) / np.sum(weights)
            pooled_se = np.sqrt(1 / np.sum(weights))
            ci_lower = pooled_mean - 1.96 * pooled_se
            ci_upper = pooled_mean + 1.96 * pooled_se
        else:
            pooled_mean = pooled_se = ci_lower = ci_upper = np.nan

        pooled[model_title] = {
            "pooled_mean": pooled_mean,
            "pooled_se": pooled_se,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper
        }

    return pooled

def meta_analysis_table(results_dict, datasets, label=""):
    """Create meta-analysis table with best model as reference."""
    # First find the best model by summed AIC
    summed = compute_summed_aic(datasets, results_dict)
    best_model = min(summed.keys(), key=lambda x: summed[x])

    pooled = meta_analytic_pooling(results_dict, datasets, best_model)

    data = []
    for model_title in model_titles:
        p = pooled[model_title]
        reliable = "Yes" if p["ci_lower"] > 0 or p["ci_upper"] < 0 else "No"
        if model_title == best_model:
            reliable = "-"
        data.append({
            "Model": model_title,
            "Pooled ΔAIC": f"{p['pooled_mean']:.2f}",
            "95% CI": f"[{p['ci_lower']:.2f}, {p['ci_upper']:.2f}]",
            "Reliably Worse?": reliable
        })

    df = pd.DataFrame(data)
    # Sort by pooled ΔAIC
    df["_sort"] = [pooled[m]["pooled_mean"] for m in df["Model"]]
    df = df.sort_values("_sort").drop("_sort", axis=1).reset_index(drop=True)

    print(f"Reference model (best by summed AIC): {best_model}\n")
    return df

In [12]:
print("=" * 60)
print("META-ANALYTIC POOLING: FREE RECALL")
print("=" * 60)
df_meta_fr = meta_analysis_table(all_results, free_recall_datasets)
print(df_meta_fr.to_string(index=False))

META-ANALYTIC POOLING: FREE RECALL
Reference model (best by summed AIC): BasePositionalCMR

              Model Pooled ΔAIC        95% CI Reliably Worse?
  BasePositionalCMR        0.00  [0.00, 0.00]               -
 BlendPositionalCMR        0.27 [-0.23, 0.76]              No
DistinctContextsCMR        0.44 [-0.38, 1.27]              No
           WeirdCMR        0.86  [0.20, 1.53]             Yes
     NoReinstateCMR        0.87  [0.27, 1.47]             Yes
  FullPositionalCMR        1.63  [1.32, 1.94]             Yes


In [13]:
print("=" * 60)
print("META-ANALYTIC POOLING: SERIAL RECALL")
print("=" * 60)
df_meta_sr = meta_analysis_table(all_results, serial_recall_datasets)
print(df_meta_sr.to_string(index=False))

META-ANALYTIC POOLING: SERIAL RECALL
Reference model (best by summed AIC): FullPositionalCMR

              Model Pooled ΔAIC           95% CI Reliably Worse?
  FullPositionalCMR        0.00     [0.00, 0.00]               -
  BasePositionalCMR        4.85     [0.83, 8.87]             Yes
 BlendPositionalCMR       10.81    [3.68, 17.93]             Yes
     NoReinstateCMR       98.04  [87.25, 108.82]             Yes
           WeirdCMR       99.84  [89.06, 110.62]             Yes
DistinctContextsCMR      549.85 [444.22, 655.48]             Yes


In [14]:
print("=" * 60)
print("META-ANALYTIC POOLING: ALL DATASETS")
print("=" * 60)
df_meta_all = meta_analysis_table(all_results, all_datasets)
print(df_meta_all.to_string(index=False))

META-ANALYTIC POOLING: ALL DATASETS
Reference model (best by summed AIC): BasePositionalCMR

              Model Pooled ΔAIC        95% CI Reliably Worse?
  BasePositionalCMR        0.00  [0.00, 0.00]               -
 BlendPositionalCMR        0.30 [-0.20, 0.79]              No
DistinctContextsCMR        0.47 [-0.35, 1.30]              No
     NoReinstateCMR        1.18  [0.58, 1.78]             Yes
           WeirdCMR        1.24  [0.58, 1.90]             Yes
  FullPositionalCMR        1.59  [1.28, 1.90]             Yes


## Summary

In [15]:
print("=" * 60)
print("SUMMARY: BEST MODELS BY POOLING METHOD")
print("=" * 60)
print()
print("FREE RECALL:")
print(f"  Summed AIC winner:     {df_free.iloc[0]['Model']}")
print(f"  Mean rank winner:      {df_ranks.sort_values('Mean Rank (FR)').iloc[0]['Model']}")
print()
print("SERIAL RECALL:")
print(f"  Summed AIC winner:     {df_serial.iloc[0]['Model']}")
print(f"  Mean rank winner:      {df_ranks.sort_values('Mean Rank (SR)').iloc[0]['Model']}")
print()
print("ALL DATASETS:")
print(f"  Summed AIC winner:     {df_all.iloc[0]['Model']}")
print(f"  Mean rank winner:      {df_ranks.sort_values('Mean Rank').iloc[0]['Model']}")

SUMMARY: BEST MODELS BY POOLING METHOD

FREE RECALL:
  Summed AIC winner:     BasePositionalCMR
  Mean rank winner:      BasePositionalCMR

SERIAL RECALL:
  Summed AIC winner:     FullPositionalCMR
  Mean rank winner:      FullPositionalCMR

ALL DATASETS:
  Summed AIC winner:     BasePositionalCMR
  Mean rank winner:      BasePositionalCMR


| Model                        |  Pooled ΔAIC vs Best: mean [CI] | Reliably Worse? |
|---:|:-----------------------------|-------------:|
| Instance-CMR + Trace Boost | 0.00 [0.00, 0.00] |  |
| Instance-CMR | [-1.73, 6.35]  | No |
| Standard CMR | 16.01 [8.33, 23.69]  | Yes |

| Model                        |  Pooled ΔAIC vs Best: mean [CI] | Reliably Worse? |
|---:|:-----------------------------|-------------:|
| Instance-CMR + Trace Boost | 0.00 [0.00, 0.00] |  |
| Instance-CMR | [-1.73, 6.35]  | No |
| Standard CMR | 16.01 [8.33, 23.69]  | Yes |